<a href="https://colab.research.google.com/github/nadjib-ken/T7Vr2-B5/blob/main/T7Vr2-B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 0: Problem Definition & Hypotheses (Pair T7Vr2-B5)

1. Problem Definition
We are using a Decision Tree model to predict the severity of road accidents in Addis Ababa (Slight, Serious, or Fatal) based only on the pre-crash circumstances (weather, road, driver, and vehicle info), excluding post-crash casualty variables to prevent data leakage. We will fix random_state = 42 everywhere so our results are reproducible.

2. Hypotheses (Our Predictions Before Testing)
- Hypothesis 1 (Scaling): Scaling the data (StandardScaler) will not change our Decision Tree's score, because Decision Trees use simple if/else rules instead of distances.
- Hypothesis 2 (Overfitting): Making the tree too deep (max_depth = None) will hurt test performance because the tree will memorize noise instead of learning general rules.
- Hypothesis 3 (Rare Fatal Crashes): Because fatal accidents are rare compared to slight injuries, the model will miss most of them unless we use class_weight='balanced' or resampling.

# Step 1: Exploration (EDA)

In [3]:
import random
import numpy as np
import pandas as pd

# Step 0: Reproducibility Seed
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Step 1: Load Data & EDA
df = pd.read_csv("raw_Addis_Ababa_city_RTA.csv")

print("1. SHAPE:", df.shape)
print("\n2. DATA TYPES:\n", df.dtypes)

print("\n3. MISSING VALUES (NaN):")
missing = df.isnull().sum()
print(missing[missing > 0])

print("\n4. UNKNOWN / NA TEXT VALUES:")
for col in df.columns:
    unk = df[col].astype(str).str.strip().str.lower().isin(["unknown", "na"]).sum()
    if unk > 0:
        print(f"  {col}: {unk}")

print("\n5. DUPLICATE ROWS:", df.duplicated().sum())

target = "Accident_severity"
print("\n6. TARGET DISTRIBUTION (CLASS BALANCE):")
print(pd.DataFrame({
    "Count": df[target].value_counts(),
    "Percent": (df[target].value_counts(normalize=True) * 100).round(2)
}))

print("\n7. FEATURE INSIGHTS (Light_conditions vs Severity %):")
print((pd.crosstab(df["Light_conditions"], df[target], normalize="index") * 100).round(2))

print("\n8. NUMBER OF VEHICLES INVOLVED:")
print(df["Number_of_vehicles_involved"].value_counts().sort_index())

leakage_cols = [
    "Number_of_casualties", "Casualty_class", "Casualty_gender",
    "Age_band_of_casualty", "Work_of_casuality", "Fitness_of_casuality"
]
print("\n9. LEAKAGE COLUMNS TO DROP IN STEP 2:", leakage_cols)

1. SHAPE: (13063, 31)

2. DATA TYPES:
 Time                             str
Day_of_week                      str
Age_band_of_driver               str
Drivers_gender                   str
Educational_level                str
Vehicle_driver_relation          str
Driving_experience               str
Type_of_vehicle                  str
Owner_of_vehicle                 str
Service_year_of_vehicle          str
Defect_of_vehicle                str
Area_accident_occured            str
Lanes_or_Medians                 str
Road_allignment                  str
Types_of_Junction                str
Road_surface_type                str
Road_surface_conditions          str
Light_conditions                 str
Weather_conditions               str
Type_of_collision                str
Number_of_vehicles_involved    int64
Number_of_casualties           int64
Vehicle_movement                 str
Casualty_class                   str
Casualty_gender                  str
Age_band_of_casualty             str

# step 2: Cleaning & Encoding

In [4]:
# -------------------------------------------------------------------------
# 1. Define Noise Placeholders
# -------------------------------------------------------------------------
missing_placeholders = [
    "", " ", "na", "n/a", "nan", "null", "none", "nil", "n/d", "tbd", "tbc",
    "unk", "unknown", "missing", "blank", "-", "--", "?", ".", "*", "#n/a", "#value!"
]

# -------------------------------------------------------------------------
# 2. Exact Duplicate Removal
# -------------------------------------------------------------------------
initial_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicates removed: {initial_rows - len(df)} rows dropped (Total remaining: {len(df)})")

# -------------------------------------------------------------------------
# 3. Target & Data Leakage Isolation (G2 Classification)
# -------------------------------------------------------------------------
target_col = "Accident_severity"
leakage_cols = [
    "Casualty_class",
    "Casualty_gender",
    "Age_band_of_casualty",
    "Work_of_casuality",
    "Fitness_of_casuality"
]
exclude_of_category_grouping = [
    "Time",
    "Day_of_week"
]

# Separate Target (y) and Feature Matrix (X)
y = df[target_col]
X = df.drop(columns=[target_col] + leakage_cols)

# -------------------------------------------------------------------------
# 4. Missing Value & "Unknown" Standardization
# -------------------------------------------------------------------------
string_cols = X.select_dtypes(include=['object', 'string']).columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Standardize text columns: trim whitespace, convert to lowercase, map noise to 'unknown'
for col in string_cols:
    X[col] = X[col].astype(str).str.strip().str.lower()
    X[col] = X[col].replace(missing_placeholders, "unknown")
    X[col] = X[col].fillna("unknown")

# For numerical columns, convert noise to np.nan
for col in num_cols:
    X[col] = X[col].replace(missing_placeholders, np.nan)

# -------------------------------------------------------------------------
# 5. Rare Category Consolidation (< 0.1% Cutoff per Column)
# -------------------------------------------------------------------------
threshold = 0.001 * len(X)  # 0.1% cutoff per column (~13 occurrences)

# exclude time and day columns from rare category consolidation

for col in string_cols.difference(exclude_of_category_grouping):
    counts = X[col].value_counts()
    # Identify categories occurring fewer than ~13 times in this column (ignoring 'unknown')
    rare_categories = counts[(counts < threshold) & (counts.index != 'unknown')].index
    
    if len(rare_categories) > 0:
        X[col] = X[col].replace(rare_categories, "other")

# -------------------------------------------------------------------------
# 6. Recombine & Export Single Deliverable (`dataset_clean.csv`)
# -------------------------------------------------------------------------
df_clean = X.copy()
df_clean[target_col] = y

df_clean.to_csv("T7Vr2-B5_dataset_clean.csv", index=False)
print("Cleaning completed successfully! Output saved to 'dataset_clean.csv'.")

Duplicates removed: 0 rows dropped (Total remaining: 13063)
Cleaning completed successfully! Output saved to 'dataset_clean.csv'.


# Step 3 - Reference splits

In [ ]:
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold

# -------------------------------------------------------------------------
# 1. Load Clean Dataset from Step 2
# -------------------------------------------------------------------------
df_clean = pd.read_csv("T7Vr2-B5_dataset_clean.csv")

target_col = "Accident_severity"
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

# -------------------------------------------------------------------------
# 2. Perform 80/20 Stratified Holdout Reference Split
# -------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=RANDOM_STATE
)

# -------------------------------------------------------------------------
# 3. Assign 'split_ref' Column & Save Updated 'dataset_clean.csv'
# -------------------------------------------------------------------------
df_clean['split_ref'] = 'train'
df_clean.loc[X_test.index, 'split_ref'] = 'test'

# Overwrite T7Vr2-B5_dataset_clean.csv to include the mandatory split_ref column
df_clean.to_csv("T7Vr2-B5_dataset_clean.csv", index=False)
print("Updated 'T7Vr2-B5_dataset_clean.csv' with 'split_ref' column successfully!\n")

# Verify row counts and percentages
print(f"Total dataset: {len(df_clean)} rows")
print(f"Train set:     {len(X_train)} rows ({len(X_train)/len(df_clean):.1%})")
print(f"Test set:      {len(X_test)} rows ({len(X_test)/len(df_clean):.1%})")

cv_reference = RepeatedStratifiedKFold(
    n_splits=5, 
    n_repeats=3, 
    random_state=42
)

print("\nRepeatedStratifiedKFold (5 folds x 3 repeats) ready for model training!")

Updated 'T7Vr2-B5_dataset_clean.csv' with 'split_ref' column successfully!

Total dataset: 13063 rows
Train set:     10450 rows (80.0%)
Test set:      2613 rows (20.0%)

RepeatedStratifiedKFold (5 folds x 3 repeats) ready for model training!
